# ML-08 - Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/8-BitRhyon/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

Lane 4 (provisional): CTR and engagement opportunity scoring for content review.
This notebook trains the first real models on the warehouse frame from ML-04 and compares them
with the value rule from the lane framing, on the same split and the same metric.

- Features: March 2026, five columns, identical to the data contract
- Label: April 2026 CTR below the March CTR median of the page's position tier
- Split: grouped by client, seed 42
- Metric: precision@K, with K = 140 reviews from the ML-02 capacity assumption

The headline is not "the model wins". It is that the hand rule is very hard to beat at the top of
the queue, and the model's real gain shows up in the ordering, not in the 140 slots.


## 1. Method choice and why

The task is ranking, so every method is judged as a scorer: train it, rank the test pages, cut at
K, and measure how many of the K are true positives.

| Method | Why it is in the comparison |
|---|---|
| **Value rule** (baseline) | The hand rule from ML-03: `(tier median CTR - page CTR) x impressions`. It already encodes the position-tier interaction, so it is a strong baseline, not a strawman. |
| **Logistic regression** | The readable linear reference. If a straight line is enough, nothing more complex is justified. |
| **Decision tree, depth 3** | A readable non-linear reference. The printed rules show what the model can express. |
| **Random forest** | A stable ensemble for tabular data, resistant to a single split. |
| **Gradient boosting** | The strongest candidate for this shape: tabular rows, non-linear interactions, 27k training rows, no text or sequence. |

Why this family: the rows are tabular, the signal is an interaction between CTR and position, and
the data is small enough that a tree ensemble trains in seconds. There is no sequence, image, or
text input that would justify a neural model.

The comparison is deliberately broader than "one model". The card asks whether complexity earns
its place, and that question needs simpler models in the table.


In [1]:
import os
import sys

import duckdb
import numpy as np
import pandas as pd
import sklearn

from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text

# Token order: env var -> Colab Secret -> local hf login. Never printed.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    try:
        from huggingface_hub import get_token
        HF_TOKEN = get_token()
    except Exception:
        HF_TOKEN = None

assert HF_TOKEN, "No Hugging Face token. Set a Colab Secret named HF_TOKEN, or run `hf auth login` once."

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

BASE = "hf://datasets/FlyRank/internship-warehouse"
FEATURES = ["ctr_pct", "avg_pos", "march_impressions_log", "position_sd", "content_age_days"]
LABEL = "low_ctr_next"
K = 140
SEED = 42

print("scikit-learn", sklearn.__version__, "| duckdb", duckdb.__version__)
print("features:", len(FEATURES), "| seed:", SEED, "| K:", K)


scikit-learn 1.9.1 | duckdb 2.0.0
features: 5 | seed: 42 | K: 140


## 2. Split design

**Grouped by client, 70% train and 30% test, seed 42.**

Pages from one client share templates, topics, and traffic patterns. A random page split would
put sibling pages on both sides of the fence, and the model would look better than it is. Grouping
by `client_hash_id` answers the question that matters: does this work on a client the model has
never seen?

The label is already forward-looking (April 2026), so the time dimension is handled by the window
choice rather than by the split. Together they close the two obvious leaks: no future features, and
no client memorisation.

The same split and seed as ML-04, so the comparison is reproducible and the two notebooks describe
the same test set.


In [2]:
def month_rollup(month):
    """One row per page-month, from one daily fact partition."""
    return f"""
    WITH daily AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp,
               SUM(gsc_clicks) AS clk,
               SUM(gsc_sum_position) AS pos_sum,
               SUM(CASE WHEN gsc_avg_position > 0 THEN gsc_impressions END) AS pos_imp,
               COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS days_gsc,
               STDDEV_SAMP(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS pos_sd
        FROM read_parquet('{BASE}/fact_content_daily_performance/month={month}/data_0.parquet')
        GROUP BY 1, 2
    )
    SELECT client_hash_id, content_hash_id, imp, clk, days_gsc, pos_sd,
           pos_sum / NULLIF(pos_imp, 0) AS avg_pos,
           100.0 * clk / NULLIF(imp, 0) AS ctr_pct
    FROM daily
    ORDER BY client_hash_id, content_hash_id
    """

march = con.execute(month_rollup("2026-03")).df()
april = con.execute(month_rollup("2026-04")).df()

frame = march[(march.imp >= 500) & (march.days_gsc >= 20) & march.avg_pos.notna()].copy()
frame["tier"] = pd.cut(
    frame.avg_pos, [0, 3, 10, 20, np.inf],
    labels=["top_3", "page_1", "page_2", "deep"],
).astype(str)

# Label rule fixed on April 1 from March data only.
tier_median_ctr = frame.groupby("tier").ctr_pct.median()

april_small = april.rename(columns={"imp": "apr_imp", "ctr_pct": "april_ctr_pct"})[
    ["client_hash_id", "content_hash_id", "apr_imp", "april_ctr_pct"]
]
frame = frame.merge(april_small, on=["client_hash_id", "content_hash_id"], how="inner")
frame = frame[(frame.apr_imp >= 100) & frame.april_ctr_pct.notna()].copy()
frame[LABEL] = (frame.april_ctr_pct < frame.tier.map(tier_median_ctr)).astype(int)

dim = con.execute(
    f"SELECT client_hash_id, content_hash_id, content_created_date "
    f"FROM read_parquet('{BASE}/dim_content.parquet')"
).df()
frame = frame.merge(dim, on=["client_hash_id", "content_hash_id"], how="left")

frame["march_impressions_log"] = np.log1p(frame.imp)
frame["position_sd"] = frame.pos_sd.fillna(0)
frame["content_age_days"] = (
    pd.Timestamp("2026-03-31") - pd.to_datetime(frame.content_created_date)
).dt.days
frame["content_age_days"] = frame.content_age_days.fillna(frame.content_age_days.median())

# Stable order so the seeded split is reproducible across DuckDB runs.
frame = frame.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)

print(f"page-months: {len(frame):,} | clients: {frame.client_hash_id.nunique()}")
print(f"label base rate: {frame[LABEL].mean():.4f}")
print("March CTR median per tier:")
print(tier_median_ctr.round(4).to_string())


page-months: 58,477 | clients: 31
label base rate: 0.5719
March CTR median per tier:
tier
deep      0.0741
page_1    0.2109
page_2    0.1642
top_3     0.2322


In [3]:
X = frame[FEATURES].fillna(0)
y = frame[LABEL]
groups = frame.client_hash_id

splitter = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])
assert not train_clients & test_clients, "client overlap between train and test"

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"train: {len(train_idx):,} pages / {len(train_clients)} clients")
print(f"test:  {len(test_idx):,} pages / {len(test_clients)} clients")
print(f"test base rate: {y_test.mean():.4f}")
print(f"client overlap: {len(train_clients & test_clients)}")


train: 27,231 pages / 21 clients
test:  31,246 pages / 10 clients
test base rate: 0.5669
client overlap: 0


## 3. Train + compare vs my baseline

Same rows, same split, same metric. The baseline is the value rule from ML-03, rebuilt here with
the same formula, month, and volume floor so the comparison uses one code path.

### The bar from ML-03 needs restating

ML-03 set precision@140 at 0.80 and at least 15 points above the rule baseline. On the real label
the base rate is 0.567 and the rule reaches 0.979 at K=140, so a 15-point margin is impossible: the
ceiling is 1.000. That bar was set on a cross-sectional sketch with a 0.49 base rate and no
baseline yet.

Restated honestly: the model must beat the rule at the same K on unseen clients, and the margin
must survive per-client variation. At K=140 the margin turns out to be about one page, which is
inside noise. The model's clearer gain is in AUC, and the K-table shows where that does and does
not reach the queue.


In [4]:
def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(y_true)[order[:k]].mean())


def recall_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(y_true)[order[:k]].sum() / np.asarray(y_true).sum())


# Baseline: hand rule from the lane framing, same month and floor as the frame.
rule_score = (
    (frame.tier.map(tier_median_ctr) - frame.ctr_pct).clip(lower=0)
    * frame.imp / 100
)

fitted = {
    "logistic regression": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=1000, random_state=SEED)
    ),
    "decision tree (depth 3)": DecisionTreeClassifier(
        max_depth=3, min_samples_leaf=200, random_state=SEED
    ),
    "random forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=20, n_jobs=-1, random_state=SEED
    ),
    "gradient boosting": HistGradientBoostingClassifier(max_iter=200, random_state=SEED),
}

test_scores = {"value rule": rule_score.iloc[test_idx].to_numpy()}
for name, model in fitted.items():
    model.fit(X_train, y_train)
    test_scores[name] = model.predict_proba(X_test)[:, 1]

rows = []
for name, scores in test_scores.items():
    rows.append({
        "model": name,
        "AUC": roc_auc_score(y_test, scores),
        "P@140": precision_at_k(y_test, scores, 140),
        "R@140": recall_at_k(y_test, scores, 140),
        "P@500": precision_at_k(y_test, scores, 500),
        "P@1000": precision_at_k(y_test, scores, 1000),
        "P@5000": precision_at_k(y_test, scores, 5000),
    })

comparison = pd.DataFrame(rows).sort_values("AUC", ascending=False).reset_index(drop=True)
print(f"test base rate: {y_test.mean():.4f} | random P@140: {y_test.mean():.4f}")
print(f"recall ceiling at K=140: {140 / y_test.sum():.4f}")
comparison.round(4)


test base rate: 0.5669 | random P@140: 0.5669
recall ceiling at K=140: 0.0079


,model,AUC,P@140,R@140,P@500,P@1000,P@5000
0,gradient boosting,0.8308,0.9857,0.0078,0.978,0.957,0.9022
1,random forest,0.8304,0.9857,0.0078,0.976,0.951,0.8988
2,logistic regression,0.8018,0.8000,0.0063,0.816,0.834,0.8428
3,decision tree (depth 3),0.7911,0.8286,0.0065,0.826,0.805,0.8212
4,value rule,0.7627,0.9786,0.0077,0.972,0.960,0.8962


**Reading the table.** Three things stand out, and two of them are uncomfortable for the model.

- The **value rule is strong**: 0.7627 AUC but 0.9786 precision at 140. It already encodes the
  position-tier interaction, which is most of the signal at the top of the queue.
- **Logistic regression is worse than the rule at the top** (0.800 vs 0.979) despite a better AUC.
  A linear score cannot represent "low CTR is bad at position 2 but normal at position 30" without
  explicit interactions, so it mis-orders the top of the list.
- **Gradient boosting wins on AUC** (0.8308 vs 0.7627) and edges the rule at K=140 by one page
  (0.9857 vs 0.9786). That one page is not a business result.

At K=1000 and K=2000 the rule is still ahead, by three pages and two pages. Only at K=5000 does
the model pull clearly ahead (0.9022 vs 0.8962). So the model's better ordering is real, but the
140-slot queue does not benefit from it.

### One split is not a validation design

A one-page difference on a single holdout is not a result. The next cell repeats the grouped
split with five seeds and reports the same metric each time. If the model only wins on some
splits, the honest conclusion is that the two methods are tied at this budget.


In [5]:
stability = []
for split_seed in [42, 7, 13, 21, 99]:
    tr_s, te_s = next(
        GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=split_seed)
        .split(X, y, groups=groups)
    )
    model_s = HistGradientBoostingClassifier(max_iter=200, random_state=SEED)
    model_s.fit(X.iloc[tr_s], y.iloc[tr_s])
    scores_s = model_s.predict_proba(X.iloc[te_s])[:, 1]
    y_s = y.iloc[te_s]
    stability.append({
        "split_seed": split_seed,
        "rule_p140": precision_at_k(y_s, rule_score.iloc[te_s], 140),
        "model_p140": precision_at_k(y_s, scores_s, 140),
        "rule_auc": roc_auc_score(y_s, rule_score.iloc[te_s]),
        "model_auc": roc_auc_score(y_s, scores_s),
    })

stability = pd.DataFrame(stability)
print(stability.round(4).to_string(index=False))
print(f"\nrule  P@140: mean {stability.rule_p140.mean():.4f}, "
      f"range {stability.rule_p140.min():.4f} to {stability.rule_p140.max():.4f}")
print(f"model P@140: mean {stability.model_p140.mean():.4f}, "
      f"range {stability.model_p140.min():.4f} to {stability.model_p140.max():.4f}")
print(f"model ahead on {(stability.model_p140 > stability.rule_p140).sum()} of {len(stability)} splits")
print(f"rule AUC mean {stability.rule_auc.mean():.4f} | model AUC mean {stability.model_auc.mean():.4f}")


 split_seed  rule_p140  model_p140  rule_auc  model_auc
         42     0.9786      0.9857    0.7627     0.8308
          7     0.9500      0.9500    0.7504     0.8217
         13     0.9643      0.9929    0.7724     0.8294
         21     0.9714      0.9786    0.7336     0.8142
         99     0.9643      0.9357    0.7633     0.8019

rule  P@140: mean 0.9657, range 0.9500 to 0.9786
model P@140: mean 0.9686, range 0.9357 to 0.9929
model ahead on 3 of 5 splits
rule AUC mean 0.7565 | model AUC mean 0.8196


## 4. Errors and interpretation

**What the model leans on.** Permutation importance on the test set, measured as the AUC drop when
one column is shuffled. The top feature should be plausible, not suspiciously perfect.

**Where it is wrong.** The top 140 are almost all correct, so the interesting errors are the
handful it misses and the clients where its ranking is weaker than the rule.

**What the queue looks like.** Which position tiers fill the 140 slots, and whether that matches
the action the editor takes.


In [6]:
best_name = "gradient boosting"
best_model = fitted[best_name]
best_scores = test_scores[best_name]

imp = permutation_importance(
    best_model, X_test, y_test, scoring="roc_auc", n_repeats=10, random_state=SEED
)
importance = (
    pd.Series(imp.importances_mean, index=FEATURES)
    .sort_values(ascending=False)
    .round(4)
)
print("permutation importance (mean AUC drop when shuffled):")
print(importance.to_string())

print("\nreadable tree (depth 3), printed rules:")
print(export_text(fitted["decision tree (depth 3)"], feature_names=FEATURES, decimals=3))


permutation importance (mean AUC drop when shuffled):
ctr_pct                  0.3132
avg_pos                  0.0567
march_impressions_log    0.0232
position_sd              0.0075
content_age_days         0.0045

readable tree (depth 3), printed rules:
|--- ctr_pct <= 0.322
|   |--- ctr_pct <= 0.182
|   |   |--- avg_pos <= 19.565
|   |   |   |--- class: 1
|   |   |--- avg_pos >  19.565
|   |   |   |--- class: 1
|   |--- ctr_pct >  0.182
|   |   |--- avg_pos <= 18.466
|   |   |   |--- class: 1
|   |   |--- avg_pos >  18.466
|   |   |   |--- class: 0
|--- ctr_pct >  0.322
|   |--- ctr_pct <= 0.438
|   |   |--- march_impressions_log <= 7.665
|   |   |   |--- class: 0
|   |   |--- march_impressions_log >  7.665
|   |   |   |--- class: 0
|   |--- ctr_pct >  0.438
|   |   |--- march_impressions_log <= 7.394
|   |   |   |--- class: 0
|   |   |--- march_impressions_log >  7.394
|   |   |   |--- class: 0



In [7]:
test_frame = frame.iloc[test_idx].copy()
test_frame["y_true"] = y_test.to_numpy()
test_frame["score_model"] = best_scores
test_frame["score_rule"] = rule_score.iloc[test_idx].to_numpy()

per_client = []
for client_id, part in test_frame.groupby("client_hash_id"):
    if part.y_true.nunique() < 2:
        continue
    per_client.append({
        "client": client_id,
        "pages": len(part),
        "rule_auc": roc_auc_score(part.y_true, part.score_rule),
        "model_auc": roc_auc_score(part.y_true, part.score_model),
    })
per_client = pd.DataFrame(per_client)
per_client["model_minus_rule"] = per_client.model_auc - per_client.rule_auc

substantial = per_client[per_client.pages >= 100]
weighted_rule = (per_client.rule_auc * per_client.pages).sum() / per_client.pages.sum()
weighted_model = (per_client.model_auc * per_client.pages).sum() / per_client.pages.sum()

print(f"all clients, unweighted: model better on {(per_client.model_minus_rule > 0).sum()} of {len(per_client)}")
print(f"clients with at least 100 test pages: {len(substantial)}, model better on "
      f"{(substantial.model_minus_rule > 0).sum()} of them")
print(f"page-weighted mean AUC: rule {weighted_rule:.4f} | model {weighted_model:.4f}")
print(f"unweighted mean AUC:    rule {per_client.rule_auc.mean():.4f} | model {per_client.model_auc.mean():.4f}")
per_client.round(4).sort_values("model_minus_rule", ascending=False).reset_index(drop=True)


all clients, unweighted: model better on 8 of 10
clients with at least 100 test pages: 7, model better on 6 of them
page-weighted mean AUC: rule 0.7604 | model 0.8254
unweighted mean AUC:    rule 0.7166 | model 0.7903


,client,pages,rule_auc,model_auc,model_minus_rule
0,client_0797ff3a1fc9a6a5,4,0.7500,1.0000,0.2500
1,client_e5c2aa26a8598242,2016,0.6801,0.8241,0.1440
2,client_fef1a8f436438636,5121,0.7069,0.7793,0.0724
3,client_3197e6291363b4db,392,0.6885,0.7564,0.0679
4,client_cd12bcfd98942aa1,25,0.6623,0.7281,0.0658
5,client_23a62021009f63c4,9496,0.8044,0.8679,0.0635
6,client_73cda7b4e4f265ea,13581,0.7662,0.8196,0.0534
7,client_9958f0a7ae1df715,367,0.6903,0.7429,0.0525
8,client_d211cb07b9059bab,4,0.6667,0.6667,0.0000
9,client_400c21c81c8b46ef,240,0.7506,0.7185,-0.0320


In [8]:
top = test_frame.sort_values("score_model", ascending=False).head(K).copy()
misses = top[top.y_true == 0]

print(f"hits in the top {K}: {int(top.y_true.sum())} | misses: {len(misses)}")
print("\nqueue composition by position tier:")
print(
    top.groupby("tier")
    .agg(pages=("y_true", "size"), precision=("y_true", "mean"))
    .round(3)
    .to_string()
)

print("\nthe misses, with their April reality:")
cols = ["tier", "ctr_pct", "avg_pos", "imp", "april_ctr_pct"]
print(misses[cols].round(4).to_string(index=False) if len(misses) else "no misses")

print("\ndistribution of the label inside the queue vs the test set:")
print(pd.DataFrame({
    "queue": top.y_true.value_counts(normalize=True).rename({0: "share label 0", 1: "share label 1"}),
    "test": test_frame.y_true.value_counts(normalize=True).rename({0: "share label 0", 1: "share label 1"}),
}).round(3).to_string())


hits in the top 140: 138 | misses: 2

queue composition by position tier:
        pages  precision
tier                    
deep       12       1.00
page_1     74       1.00
page_2      4       1.00
top_3      50       0.96

the misses, with their April reality:
 tier  ctr_pct  avg_pos    imp  april_ctr_pct
top_3   0.0354   2.6316 2826.0         0.6787
top_3   0.0178   0.9811 5605.0         0.3448

distribution of the label inside the queue vs the test set:
               queue   test
y_true                     
share label 1  0.986  0.567
share label 0  0.014  0.433


**Feature reading.** The permutation table puts March CTR first by a wide margin, which is
expected: CTR persists month to month. Position is second, which is also expected because the label
is tier-relative. Position spread and content age contribute little. Nothing is suspiciously
perfect, so there is no sign of a leak in the feature list. The leak experiment in ML-04 showed
what a suspicious feature looks like.

**Error reading.** The top 140 contain two misses out of 140. Both are pages at positions 1 to 3
with a very low March CTR (0.02% to 0.04%) whose April CTR returned to 0.34% to 0.68%, above their
tier median. A page in the top three with a CTR near zero for a whole month is an outlier, and the
model treats it as an opportunity. Sometimes the month was the anomaly, not the page.

**Client reading.** The page-weighted AUC is the honest aggregate: 0.760 for the rule and 0.825
for the model. By client, the model is better on 6 of the 7 clients with at least 100 test pages.
Three clients have fewer than 30 pages, so their per-client AUC is noise; the table shows them, but
the claim does not depend on them.

**Verdict.** If the team reviews 140 pages a month, the value rule is the recommended default. The
model's AUC gain is real and survives page weighting. At K=140, though, it wins on 3 of 5 splits,
ties on one, and loses on one by four pages. That is a tie, not an improvement. Revisit the model
if the budget grows toward 5,000 pages, where the single-split table shows a larger gap.


## Self-check

- [x] Method choice explained, with simpler models in the table so complexity has to earn its place
- [x] Split design stated and enforced: grouped by client, zero client overlap asserted, and repeated across five seeds
- [x] Baseline and models scored on the same rows, the same split, and the same metrics
- [x] Useful metrics reported: AUC, precision@K at several K, recall at the budget
- [x] Features interpreted with permutation importance, and the top feature sanity-checked
- [x] Errors read: the misses, the queue composition, and the per-client variation with page weighting
- [x] The pre-registered ML-03 bar was checked, found unreachable, and restated openly
- [x] Reproducible: fixed seeds, stable row order, library versions printed
- [x] No client names, URLs, private queries, or tokens anywhere
- [x] Committed under `work/notebooks/`
